In [1]:
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from typing import List, Tuple, Optional
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs


try:
    from molgpt.fusion_model import FusionConfig, MolGPTWithRNA
    from molgpt.tokenizer import JsonSmilesTokenizer
except ImportError:
    # Если запускается из другой директории
    import sys
    sys.path.append(r"C:\Users\User\molgpt")
    from molgpt.fusion_model import FusionConfig, MolGPTWithRNA
    from molgpt.tokenizer import JsonSmilesTokenizer

In [2]:
df = pd.read_csv(r"data\Data_ML.csv")

In [3]:
df.head(5)

,ncRNA_Expression_Pattern,Sequence,SMILES,MorganFP,MorganFP_String,MolWt,LogP,NumHAcceptors,TPSA,NumRotatableBonds,embeddings
0,1,GUGAAAUGUUCAGGACCACUUG,CCOP(=S)(OCC)OC1=NC(=C(C=C1Cl)Cl)Cl,[0 0 0 ... 0 0 0],"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",350.591,4.7181,5,40.58,6,[ 5.83481900e-02 1.07824937e-01 -6.31397590e-...
1,1,UUCACUACUAGCAGAACUCGG,CCOP(=S)(OCC)OC1=NC(=C(C=C1Cl)Cl)Cl,[0 0 0 ... 0 0 0],"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",350.591,4.7181,5,40.58,6,[ 0.10364545 0.18739885 -0.05566074 -0.080716...
2,1,UGGAAGACUAGUGAUUUUGUUGUU,CCOP(=S)(OCC)OC1=NC(=C(C=C1Cl)Cl)Cl,[0 0 0 ... 0 0 0],"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",350.591,4.7181,5,40.58,6,[-4.53317612e-02 1.61736198e-02 -4.28632796e-...
3,1,UAAAGCUAGAUAACCGAAAGUA,CCOP(=S)(OCC)OC1=NC(=C(C=C1Cl)Cl)Cl,[0 0 0 ... 0 0 0],"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",350.591,4.7181,5,40.58,6,[ 0.03229173 0.10842009 -0.08173083 -0.092944...
4,1,UAAAGCUAGAUAACCGAAAGUA,[Cd],[0 0 0 ... 0 0 0],"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",112.412,-0.0025,0,0.00,0,[ 0.03229173 0.10842009 -0.08173083 -0.092944...


In [4]:
"""
код обучения и генерации для MolGPTWithRNA

"""

import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from typing import List, Tuple, Optional
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

# Импорт модели
try:
    from molgpt.fusion_model import FusionConfig, MolGPTWithRNA
    from molgpt.tokenizer import JsonSmilesTokenizer
except ImportError:
    import sys
    sys.path.append(r"C:\Users\User\molgpt")
    from fusion_model import FusionConfig, MolGPTWithRNA
    from tokenizer import JsonSmilesTokenizer


# ===================== КОНФИГУРАЦИЯ =====================

@dataclass
class Config:
    # Пути
    tok_json: Path = Path(r"C:\Users\User\molgpt\moses2_stoi.json")
    ckpt_init: Path = Path(r"C:\Users\User\Desktop\ITMO\DIPLOMA\weights\char_rnn_10percent_gua.pt") 
    data_csv: Path = Path(r"data\Data_ML.csv")
    save_ckpt: Path = Path(r"weights\trained_model.pt")
    
    # Гиперпараметры
    block_size: int = 116
    batch_size: int = 16
    epochs: int = 1
    lr: float = 3e-4
    val_split: float = 0.10
    seed: int = 42
    weight_decay: float = 0.01
    clip_norm: float = 1.0
    
    # Генерация
    max_new_tokens: int = 100
    temperature: float = 0.8
    top_k: int = 50
    top_p: float = 0.9
    
    # Колонки
    seq_column: str = "Sequence"
    smiles_column: str = "SMILES"
    label_column: str = "ncRNA_Expression_Pattern"
    
    device: torch.device = None
    
    def __post_init__(self):
        if self.device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ===================== УТИЛИТЫ =====================

def set_seed(seed: int):
    """Фиксируем все сиды для воспроизводимости."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


from rdkit import Chem
from typing import List, Dict, Any, Optional
import numpy as np


def is_valid_smiles(s: str) -> bool:
    """Проверка валидности SMILES через RDKit. Возвращает True/False."""
    if not isinstance(s, str):
        return False
    
    smiles = s.strip()
    if not smiles:
        return False
    
    mol = Chem.MolFromSmiles(smiles)
    return mol is not None


def smiles_to_fp(s: str, radius: int = 2, n_bits: int = 2048):
    """
    Вычисление Morgan fingerprint.
    Возвращает объект для сравнения через DataStructs.TanimotoSimilarity
    """
    mol = Chem.MolFromSmiles(s) if isinstance(s, str) and s.strip() else None
    if mol is None:
        return None
    try:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
        return fp
    except Exception:
        return None


def tanimoto_similarity(fp1, fp2) -> float:
    """Вычисление Tanimoto similarity между двумя fingerprint'ами."""
    if fp1 is None or fp2 is None:
        return 0.0
    return DataStructs.TanimotoSimilarity(fp1, fp2)



ckpt_path = Config.save_ckpt if Config.save_ckpt.exists() else Config.ckpt_init
print(f"[CHECK] save_ckpt exists: {Config.save_ckpt.exists()}")
print(f"[CHECK] ckpt_init exists: {Config.ckpt_init.exists()}")
print(f"[CHECK] Will load from: {ckpt_path}")
print(f"[CHECK] save_ckpt size: {Config.save_ckpt.stat().st_size / 1024 / 1024:.1f} MB")
# ===================== ЗАГРУЗКА МОДЕЛИ =====================

def load_model_with_resize(
    tok_json: Path,
    ckpt_path: Path,
    device: torch.device,
    block_size: int,
    strict: bool = False
) -> Tuple[JsonSmilesTokenizer, MolGPTWithRNA]:
    """
    Загружает токенизатор и модель с resize vocabulary.
    """
    # Загружаем токенизатор
    tokenizer = JsonSmilesTokenizer(str(tok_json))
    stoi = tokenizer.stoi if hasattr(tokenizer, "stoi") else tokenizer.vocab
    
    pad_id = stoi["<pad>"]
    bos_id = stoi["<bos>"]
    eos_id = stoi["<eos>"]
    vocab_size = len(stoi)
    
    print(f"[tokenizer] vocab_size={vocab_size}, pad={pad_id}, bos={bos_id}, eos={eos_id}")
    
    # Создаем конфиг
    cfg = FusionConfig(
        vocab_size=vocab_size,
        pad_id=pad_id,
        block_size=block_size,
        n_layer=8,
        n_head=8,
        n_embd=256,
        cond_dim=256,
        num_props=5,  # Оставляем для совместимости, но не используем
        dropout=0.1,
        cross_attn_from=4,
        prefix_len=10,
        rna_fm_trainable_layers=3,
        use_props_in_generate=False,  # не используем свойства при генерации
    )
    
    model = MolGPTWithRNA(cfg, freeze_rna=True)
    
    # Загружаем чекпоинт с resize
    ckpt = torch.load(str(ckpt_path), map_location="cpu", weights_only=False)
    state = ckpt.get("model", ckpt)
    model_state = model.state_dict()
    
    new_state = {}
    for k, v in state.items():
        if k not in model_state:
            continue
            
        if v.shape == model_state[k].shape:
            new_state[k] = v
        elif v.dim() == 2 and model_state[k].dim() == 2 and v.shape[1] == model_state[k].shape[1]:
            # Resize для embedding и lm_head (vocab различается)
            v_old, dim = v.shape
            v_new, _ = model_state[k].shape
            new_v = model_state[k].clone()
            min_v = min(v_old, v_new)
            new_v[:min_v, :] = v[:min_v, :]
            
            # Инициализация новых токенов
            if v_new > v_old:
                std = 1.0 / math.sqrt(dim)
                new_v[v_old:v_new, :].normal_(0.0, std)
            new_state[k] = new_v
            print(f"[resize] {k}: {v.shape} -> {model_state[k].shape}")
        elif v.dim() == 1 and model_state[k].dim() == 1:
            # Для bias и других 1D тензоров
            v_old = v.shape[0]
            v_new = model_state[k].shape[0]
            new_v = model_state[k].clone()
            min_v = min(v_old, v_new)
            new_v[:min_v] = v[:min_v]
            new_state[k] = new_v
            print(f"[resize] {k}: {v.shape} -> {model_state[k].shape}")
    
    model_state.update(new_state)
    model.load_state_dict(model_state, strict=strict)
    model.to(device)
    
  
    model.config.use_props_in_generate = False
    
    print(f"[model] loaded on {device}")
    return tokenizer, model


# ===================== DATASET =====================

class RNAMolDataset(Dataset):
    """Dataset для обучения на микроРНК и молекулах."""
    
    def __init__(
        self,
        df: pd.DataFrame,
        tokenizer: JsonSmilesTokenizer,
        block_size: int,
        seq_column: str = "Sequence",
        smiles_column: str = "SMILES",
        label_column: str = "ncRNA_Expression_Pattern",
    ):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.block_size = block_size
        self.seq_column = seq_column
        self.smiles_column = smiles_column
        self.label_column = label_column
        
        stoi = tokenizer.stoi if hasattr(tokenizer, "stoi") else tokenizer.vocab
        self.pad_id = stoi["<pad>"]
        self.bos_id = stoi["<bos>"]
        self.eos_id = stoi["<eos>"]
        
    def _encode_smiles(self, smiles: str) -> torch.Tensor:
        """Кодирует SMILES в тензор фиксированной длины."""
        ids = self.tokenizer.encode(smiles)
        ids = [self.bos_id] + ids + [self.eos_id]
        
        if len(ids) > self.block_size:
            ids = ids[:self.block_size]
        else:
            ids = ids + [self.pad_id] * (self.block_size - len(ids))
            
        return torch.tensor(ids, dtype=torch.long)
    
    def __len__(self) -> int:
        return len(self.df)
    
    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        
        # Кодируем SMILES
        x = self._encode_smiles(str(row[self.smiles_column]))
        
   
        y = torch.cat([x[1:], torch.tensor([self.pad_id], dtype=torch.long)])
        
        # Метка связывания
        bind = torch.tensor(int(row[self.label_column]), dtype=torch.long)
        
        # Последовательность РНК
        seq = str(row[self.seq_column]).strip()
        
        return {
            "input_ids": x,
            "target": y,
            "bind": bind,
            "sequence": seq,
        }


# ===================== COLLATE FUNCTION =====================

def collate_fn(
    batch: List[dict],
    alphabet,  # RNA-FM alphabet
    pad_id: int,
) -> dict:
    """
    Собирает батч
    """
    input_ids = torch.stack([b["input_ids"] for b in batch])
    target = torch.stack([b["target"] for b in batch])
    bind = torch.stack([b["bind"] for b in batch])
    
    # Подготовка РНК через RNA-FM batch_converter
    # Заменяем T на U (DNA -> RNA)
    seqs = [("rna", b["sequence"].replace("T", "U")) for b in batch]
    batch_converter = alphabet.get_batch_converter()
    _, _, rna_tokens = batch_converter(seqs)
    
    # Маска для РНК (1 = реальный токен, 0 = паддинг)
    rna_mask = (rna_tokens != alphabet.padding_idx).long()
    
    return {
        "input_ids": input_ids,
        "target": target,
        "bind": bind,
        "rna_tokens": rna_tokens,
        "rna_mask": rna_mask,
        "ignore_index": pad_id,
    }



[CHECK] save_ckpt exists: True
[CHECK] ckpt_init exists: True
[CHECK] Will load from: weights\trained_model.pt
[CHECK] save_ckpt size: 416.5 MB


In [5]:
# ===================== ОБУЧЕНИЕ =====================

def train_epoch(
    model: MolGPTWithRNA,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    clip_norm: float,
    epoch: int,
) -> float:
    """Одна эпоха обучения."""
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for batch_idx, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(device)
        target = batch["target"].to(device)
        bind = batch["bind"].to(device)
        rna_tokens = batch["rna_tokens"].to(device)
        rna_mask = batch["rna_mask"].to(device)
        
        optimizer.zero_grad()
        
       
        # Передаем dummy props (нули), но use_props=False значит они не используются
        batch_size = input_ids.size(0)
        dummy_props = torch.zeros(batch_size, model.config.num_props, device=device)
        
        logits, loss, bind_logits, _ = model(
            idx=input_ids,
            target=target,
            token_type_ids=torch.ones_like(input_ids),  # тип 1 = молекула
            rna_tokens=rna_tokens,
            rna_mask=rna_mask,
            prop=dummy_props,
            bind_labels=bind,
            use_props=False,  
        )
        
        if loss is None:
            # Если вдруг loss не посчитался
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                target.view(-1),
                ignore_index=model.pad_id,
            )
        
        # Добавляем loss для предсказания связывания (опционально, для стабильности)
        bind_loss = F.cross_entropy(bind_logits, bind)
        total_loss_batch = loss + 0.1 * bind_loss
        
        total_loss_batch.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        
        if batch_idx % 100 == 0:
            print(f"  [train] epoch {epoch}, batch {batch_idx}/{len(loader)}, loss: {loss.item():.4f}")
    
    return total_loss / num_batches


@torch.no_grad()
def validate(
    model: MolGPTWithRNA,
    loader: DataLoader,
    device: torch.device,
) -> float:
    """Валидация."""
    model.eval()
    total_loss = 0.0
    num_batches = 0
    
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        target = batch["target"].to(device)
        bind = batch["bind"].to(device)
        rna_tokens = batch["rna_tokens"].to(device)
        rna_mask = batch["rna_mask"].to(device)
        
        batch_size = input_ids.size(0)
        dummy_props = torch.zeros(batch_size, model.config.num_props, device=device)
        
        logits, loss, _, _ = model(
            idx=input_ids,
            target=target,
            token_type_ids=torch.ones_like(input_ids),
            rna_tokens=rna_tokens,
            rna_mask=rna_mask,
            prop=dummy_props,
            bind_labels=bind,
            use_props=False,
        )
        
        if loss is not None:
            total_loss += loss.item()
            num_batches += 1
    
    return total_loss / num_batches if num_batches > 0 else float('inf')


def train(config: Config):
    """Полный цикл обучения."""
    set_seed(config.seed)
    
    # Проверяем пути
    assert config.tok_json.exists(), f"Tokenizer not found: {config.tok_json}"
    assert config.ckpt_init.exists(), f"Checkpoint not found: {config.ckpt_init}"
    assert config.data_csv.exists(), f"Data not found: {config.data_csv}"
    
    # Загружаем модель
    tokenizer, model = load_model_with_resize(
        config.tok_json,
        config.ckpt_init,
        config.device,
        config.block_size,
    )
    
    # Загружаем данные
    df = pd.read_csv(config.data_csv)
    
    # Очистка данных
    df = df.dropna(subset=[config.seq_column, config.smiles_column, config.label_column])
    df[config.label_column] = pd.to_numeric(
        df[config.label_column], errors="coerce"
    ).round().clip(0, 1).fillna(0).astype(int)
    
    print(f"[data] loaded {len(df)} samples")
    print(f"[data] bind=1: {(df[config.label_column] == 1).sum()}, bind=0: {(df[config.label_column] == 0).sum()}")
    
    # Сплит с стратификацией
    train_df, val_df = train_test_split(
        df,
        test_size=config.val_split,
        random_state=config.seed,
        stratify=df[config.label_column],
    )
    
    print(f"[split] train: {len(train_df)}, val: {len(val_df)}")
    
    # Dataset и DataLoader
    train_ds = RNAMolDataset(train_df, tokenizer, config.block_size, config.seq_column, config.smiles_column, config.label_column)
    val_ds = RNAMolDataset(val_df, tokenizer, config.block_size, config.seq_column, config.smiles_column, config.label_column)
    
    stoi = tokenizer.stoi if hasattr(tokenizer, "stoi") else tokenizer.vocab
    pad_id = stoi["<pad>"]
    
    train_loader = DataLoader(
        train_ds,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=0,
        collate_fn=lambda batch: collate_fn(batch, model.rna.alphabet, pad_id),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=lambda batch: collate_fn(batch, model.rna.alphabet, pad_id), 
    )
    
    # Оптимизатор
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.lr,
        weight_decay=config.weight_decay,
    )
    
    # Цикл обучения
    best_val_loss = float('inf')
    
    for epoch in range(1, config.epochs + 1):
        print(f"\n{'='*50}")
        print(f"Epoch {epoch}/{config.epochs}")
        print(f"{'='*50}")
        
        train_loss = train_epoch(model, train_loader, optimizer, config.device, config.clip_norm, epoch)
        val_loss = validate(model, val_loader, config.device)
        
        print(f"[epoch {epoch}] train_loss: {train_loss:.4f}, val_loss: {val_loss:.4f}")
        
        # Сохраняем лучшую модель
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_dict = {
                "model": model.state_dict(),
                "config": model.config,
                "epoch": epoch,
                "val_loss": val_loss,
            }
            config.save_ckpt.parent.mkdir(parents=True, exist_ok=True)
            torch.save(save_dict, config.save_ckpt)
            print(f"[save] saved best model to {config.save_ckpt}")
    
    print("\n[done] Training completed!")
    return model, tokenizer


In [8]:
# ===================== ГЕНЕРАЦИЯ =====================

from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from typing import List, Dict, Any, Tuple, Optional
import numpy as np
import pandas as pd
import torch
#model = MolGPTWithRNA(cfg, freeze_rna=True)

def is_valid_smiles(s: str, min_atoms: int = 10) -> bool:
    """Проверка валидности + минимальный размер."""
    if not isinstance(s, str):
        return False
    
    smiles = s.strip()
    if not smiles:
        return False
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    
    # Фильтр по размеру (убираем ацетаты, формиаты и т.п.)
    if mol.GetNumAtoms() < min_atoms:
        return False
    
    return True
    


def smiles_to_fp(smiles: str, radius: int = 2, n_bits: int = 2048) -> Optional[np.ndarray]:
    """ECFP fingerprint через RDKit."""
    if smiles is None or not is_valid_smiles(smiles):
        return None
    
    mol = Chem.MolFromSmiles(smiles)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)


def tanimoto_similarity(fp1: np.ndarray, fp2: np.ndarray) -> float:
    """Tanimoto similarity между двумя fingerprints."""
    bv1 = DataStructs.ExplicitBitVect(len(fp1))
    bv2 = DataStructs.ExplicitBitVect(len(fp2))
    
    for i, bit in enumerate(fp1):
        if bit:
            bv1.SetBit(i)
    for i, bit in enumerate(fp2):
        if bit:
            bv2.SetBit(i)
    
    return DataStructs.TanimotoSimilarity(bv1, bv2)


def build_reference_fingerprints(
    df: pd.DataFrame, 
    smiles_column: str = "SMILES",
    label_column: str = "ncRNA_Expression_Pattern",
    bind_value: int = 1  # <-- Только позитивные по умолчанию
) -> List:
    """Строит fingerprint'ы для датасета, фильтруя по метке связывания."""
    fps = []
    # Фильтруем только нужные метки
    df_filtered = df[df[label_column] == bind_value]
    
    for smiles in df_filtered[smiles_column].astype(str):
        if is_valid_smiles(smiles):
            fp = smiles_to_fp(smiles)
            if fp is not None:
                fps.append(fp)
    
    print(f"[ref] Built {len(fps)} reference fingerprints (bind={bind_value})")
    return fps


class MoleculeGenerator:
    """Класс для генерации молекул с валидацией и оценкой."""
    
    def __init__(
        self,
        model,
        tokenizer,
        device: torch.device,
        block_size: int,
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.block_size = block_size
        
        stoi = tokenizer.stoi if hasattr(tokenizer, "stoi") else tokenizer.vocab
        self.pad_id = stoi["<pad>"]
        self.bos_id = stoi["<bos>"]
        self.eos_id = stoi["<eos>"]
        self.unk_id = stoi["<unk>"]
        
        self.model.eval()
        
        # Подготовка batch_converter для РНК
        self.batch_converter = model.rna.alphabet.get_batch_converter()
    
    def prepare_rna(self, sequence: str) -> Tuple[torch.Tensor, torch.Tensor]:
        """Подготавливает токены РНК."""
        # DNA -> RNA, убираем пробелы
        seq_clean = str(sequence).strip().replace("T", "U").replace(" ", "")
        _, _, tokens = self.batch_converter([("rna", seq_clean)])
        tokens = tokens.to(self.device)
        mask = (tokens != self.model.rna.alphabet.padding_idx).long().to(self.device)
        return tokens, mask
    
    @torch.no_grad()
    def generate(
        self,
        rna_sequence: str,
        bind_label: int = 1,
        n_samples: int = 10,
        max_new_tokens: int = 100,
        temperature: float = 0.8,
        top_k: int = 50,
        top_p: float = 0.9,
    ) -> List[str]:
        """
        Генерирует молекулы для заданной микроРНК.
        """
        # Подготавливаем РНК
        rna_tokens, rna_mask = self.prepare_rna(rna_sequence)
        
        # Расширяем до батча нужного размера
        rna_tokens = rna_tokens.repeat(n_samples, 1)
        rna_mask = rna_mask.repeat(n_samples, 1)
        
        # Начальные токены (BOS)
        idx = torch.full((n_samples, 1), self.bos_id, dtype=torch.long, device=self.device)
        token_type_ids = torch.ones_like(idx)
        
        # Dummy properties
        dummy_props = torch.zeros(n_samples, self.model.config.num_props, device=self.device)
        bind_labels = torch.full((n_samples,), bind_label, dtype=torch.long, device=self.device)
        
        # Генерация через model.generate()
        generated = self.model.generate(
            idx=idx,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k if top_k > 0 else None,
            token_type_ids=token_type_ids,
            rna_tokens=rna_tokens,
            rna_mask=rna_mask,
            prop=dummy_props,
            bind_labels=bind_labels,
            use_props=False,
        )
        
        # Декодируем
        molecules = []
        for seq_ids in generated:
            ids = seq_ids.tolist()
            # Находим EOS и обрезаем
            if self.eos_id in ids:
                ids = ids[:ids.index(self.eos_id)]
            # Убираем специальные токены
            clean_ids = [i for i in ids if i not in (self.bos_id, self.pad_id, self.unk_id)]
            smiles = self.tokenizer.decode(clean_ids).strip()
            molecules.append(smiles)
        
        return molecules
    
    def validate_molecules(
        self,
        molecules: List[str],
        reference_fps: Optional[List] = None,
        #similarity_threshold: float = 0.5,
    ) -> Dict[str, Any]:
        """
        Проверяет валидность молекул через RDKit.
        """
        results = {
            "molecules": [],
            "valid": [],
            "fingerprints": [],
            "similarities": [],
        }
        
        for smiles in molecules:
            results["molecules"].append(smiles)
            
            # Простая валидация: True/False
            is_valid = is_valid_smiles(smiles)
            results["valid"].append(is_valid)
            
            # Fingerprint только для валидных
            fp = smiles_to_fp(smiles) if is_valid else None
            results["fingerprints"].append(fp)
            
            # Сходство с референсом
            if reference_fps is not None and fp is not None:
                similarities = [tanimoto_similarity(fp, ref_fp) for ref_fp in reference_fps]
                max_sim = max(similarities) if similarities else 0.0
                results["similarities"].append(max_sim)
            else:
                results["similarities"].append(None)
        
        # Статистика
        valid_count = sum(results["valid"])
        print(f"\n[validation] Valid: {valid_count}/{len(molecules)} ({100*valid_count/len(molecules):.1f}%)")
        
        if reference_fps:
            valid_sims = [s for s in results["similarities"] if s is not None]
            if valid_sims:
                print(f"[validation] Avg Tanimoto similarity: {np.mean(valid_sims):.3f}")
                print(f"[validation] Max similarity: {np.max(valid_sims):.3f}")
        
            
        return results


def generate_for_dataset_rna(
    config,
    n_samples_per_rna: int = 10,
    n_rnas: int = 5,
):
    """Генерирует молекулы для микроРНК из датасета."""
    # Загружаем модель
    tokenizer, model = load_model_with_resize(
        config.tok_json,
        config.save_ckpt if config.save_ckpt.exists() else config.ckpt_init,
        config.device,
        config.block_size,
    )
    
    # Загружаем данные для референсных fingerprint
    df = pd.read_csv(config.data_csv)
    df = df.dropna(subset=[config.seq_column, config.smiles_column])
    
    reference_fps = build_reference_fingerprints(
    df, 
    config.smiles_column, 
    config.label_col, 
    bind_value=1  
)
    
    # Генератор
    generator = MoleculeGenerator(model, tokenizer, config.device, config.block_size)
    
    # Берем первые n_rnas уникальных последовательностей с bind=1
    test_rnas = df[df[config.label_column] == 1][config.seq_column].unique()[:n_rnas]
    
    all_results = []
    
    for i, rna_seq in enumerate(test_rnas):
        print(f"\n{'='*60}")
        print(f"RNA {i+1}/{len(test_rnas)}: {rna_seq[:50]}...")
        print(f"{'='*60}")
        
        # Генерируем
        mols = generator.generate(
            rna_sequence=rna_seq,
            bind_label=1,
            n_samples=n_samples_per_rna,
            max_new_tokens=config.max_new_tokens,
            temperature=config.temperature,
            top_k=config.top_k,
            top_p=config.top_p,
        )
        
        # Валидируем
        results = generator.validate_molecules(mols, reference_fps)
        all_results.append({
            "rna": rna_seq,
            "results": results,
        })
        
        # Выводим примеры
        print(f"\n[generated molecules]")
        for j, (mol, valid, sim) in enumerate(zip(
            results["molecules"],
            results["valid"],
            results["similarities"],
        )):
            status = "✓" if valid else "✗"
            sim_str = f"{sim:.3f}" if sim else "N/A"
            print(f"  {j+1}. {status} {mol[:60]:60s} | sim: {sim_str}")
    
    ckpt_path = config.save_ckpt if config.save_ckpt.exists() else config.ckpt_init
    print(f"[load] Loading from: {ckpt_path}")
    tokenizer, model = load_model_with_resize(...)

    return all_results


def generate_for_custom_rna(
    config,
    rna_sequence: str,
    n_samples: int = 10,
):
    """Генерирует молекулы для произвольной микроРНК"""
    tokenizer, model = load_model_with_resize(
        config.tok_json,
        config.save_ckpt if config.save_ckpt.exists() else config.ckpt_init,
        config.device,
        config.block_size,
    )
    
    # Загружаем референсные fingerprint
    df = pd.read_csv(config.data_csv)
    reference_fps = build_reference_fingerprints(
    df, 
    config.smiles_column, 
    config.label_column, 
    bind_value=1
)
    
    generator = MoleculeGenerator(model, tokenizer, config.device, config.block_size)
    
    print(f"\n[custom RNA] {rna_sequence[:60]}...")
    
    mols_bind1 = generator.generate(
        rna_sequence=rna_sequence,
        bind_label=1,
        n_samples=n_samples,
        max_new_tokens=config.max_new_tokens,
        temperature=config.temperature,
        top_k=config.top_k,
        top_p=config.top_p,
    )
    
    mols_bind0 = generator.generate(
        rna_sequence=rna_sequence,
        bind_label=0,
        n_samples=n_samples,
        max_new_tokens=config.max_new_tokens,
        temperature=config.temperature,
        top_k=config.top_k,
        top_p=config.top_p,
    )

    print("=== bind=1 ===")
    for m in mols_bind1[:5]:
        print(f"  {m}")

    print("=== bind=0 ===")
    for m in mols_bind0[:5]:
        print(f"  {m}")
    
    results = generator.validate_molecules(mols, reference_fps)
    
    print(f"\n[results]")
    for mol, valid, sim in zip(
        results["molecules"],
        results["valid"],
        results["similarities"],
    ):
        status = "✓ VALID" if valid else "✗ INVALID"
        sim_str = f"{sim:.3f}" if sim else "N/A"
        print(f"  {status} | {mol[:60]:60s} | sim: {sim_str}")
    
    return results

cfg = Config()
tokenizer, model = load_model_with_resize(
    cfg.tok_json,
    cfg.save_ckpt if cfg.save_ckpt.exists() else cfg.ckpt_init,
    cfg.device,
    cfg.block_size,
)
# После загрузки модели:
with torch.no_grad():
    emb0 = model.bind_emb(torch.tensor([0]))
    emb1 = model.bind_emb(torch.tensor([1]))
    print(f"bind=0: {emb0[:5]}")  # Первые 5 значений
    print(f"bind=1: {emb1[:5]}")
    print(f"Разница (L2): {torch.norm(emb0 - emb1).item()}")

[tokenizer] vocab_size=58, pad=26, bos=27, eos=28
[model] loaded on cpu
bind=0: tensor([[-1.5965e+00, -7.0593e-01, -6.4455e-01,  5.4387e-01, -3.5525e-01,
          1.8877e-02,  2.9389e-04, -1.1635e+00, -1.0811e+00,  9.9043e-01,
          5.4128e-01,  6.5095e-01,  1.2171e+00, -7.8957e-01,  2.1175e-01,
          5.5239e-01,  1.0296e+00,  1.1934e+00, -5.1938e-01,  3.6063e-01,
         -6.9067e-01,  4.6139e-01, -2.0018e+00,  5.6161e-02, -1.9344e-01,
         -1.1948e-01, -1.6449e+00, -2.0611e+00,  1.4871e+00,  6.8821e-01,
          1.0604e+00,  1.5209e+00,  1.8996e+00,  6.8908e-01, -7.1086e-01,
         -3.9076e-01,  2.5803e-01,  7.0445e-01,  5.9583e-01,  5.5063e-01,
          3.5555e-01, -1.3129e-01, -9.5099e-01, -4.4044e-01, -2.6350e-01,
         -7.3819e-01, -8.1638e-01,  1.4616e-01, -1.1132e+00, -6.2741e-01,
          2.3991e-01, -7.0054e-01,  3.7322e-01, -1.2130e+00,  1.7422e+00,
         -3.3058e-01, -3.8324e-01,  2.5456e-01,  1.4843e+00, -3.8342e-01,
         -1.5591e+00, -8.9753e-0

In [9]:
print(f"Checkpoint exists: {Config.save_ckpt.exists()}")
print(f"Loading from: {ckpt_path}")

Checkpoint exists: True
Loading from: weights\trained_model.pt


In [ ]:

if __name__ == "__main__":
    config = Config()
    
    print("="*60)
    print("MolGPT with microRNA - Training and Generation")
    print(f"Device: {config.device}")
    print("="*60)
    
    # Режим 1: Обучение
    print("\n[mode] Training")
    model, tokenizer = train(config)
    
    # Режим 2: Генерация для РНК из датасета
    print("\n[mode] Generation for dataset RNAs")
    results = generate_for_dataset_rna(config, n_samples_per_rna=30, n_rnas=10)
    
    # Режим 3: Генерация для произвольной РНК
    print("\n[mode] Generation for custom RNA")
    custom_rna = "GUGAAAUGUUCAGGACCACUUG" 
    custom_results = generate_for_custom_rna(config, custom_rna, n_samples=50)

MolGPT with microRNA - Training and Generation
Device: cpu

[mode] Training
[tokenizer] vocab_size=58, pad=26, bos=27, eos=28
[model] loaded on cpu
[data] loaded 30442 samples
[data] bind=1: 16580, bind=0: 13862
[split] train: 27397, val: 3045

Epoch 1/1
  [train] epoch 1, batch 0/1713, loss: 4.4602
  [train] epoch 1, batch 100/1713, loss: 1.5059
  [train] epoch 1, batch 200/1713, loss: 1.2593
  [train] epoch 1, batch 300/1713, loss: 1.2356
  [train] epoch 1, batch 400/1713, loss: 1.0023
  [train] epoch 1, batch 500/1713, loss: 1.1529
  [train] epoch 1, batch 600/1713, loss: 0.9908
  [train] epoch 1, batch 700/1713, loss: 0.9750
  [train] epoch 1, batch 800/1713, loss: 0.7204
  [train] epoch 1, batch 900/1713, loss: 0.9032
  [train] epoch 1, batch 1000/1713, loss: 0.6499
  [train] epoch 1, batch 1100/1713, loss: 0.7830
  [train] epoch 1, batch 1200/1713, loss: 0.7318
  [train] epoch 1, batch 1300/1713, loss: 0.7360
  [train] epoch 1, batch 1400/1713, loss: 0.7947
  [train] epoch 1, bat

[18:59:17] WARNING: not removing hydrogen atom without neighbors
[18:59:17] WARNING: not removing hydrogen atom without neighbors


[ref] Built 16029 reference fingerprints (bind=1)

RNA 1/10: GUGAAAUGUUCAGGACCACUUG...


[18:59:34] Explicit valence for atom # 16 N, 5, is greater than permitted
[18:59:55] SMILES Parse Error: unclosed ring for input: 'C1=CC=C(C(=C1O)O)C2=COC3=C2OC=C4C(=O)C=CC=C5'
[19:00:03] SMILES Parse Error: extra open parentheses for input: 'CC(C)S(=O)(=O)NC1=C(C(=C(C=C1)F)C(=O)C2=CNC3=C2C=CC=C3C(=O)OC4=CC=CC=C4'
[19:00:12] Explicit valence for atom # 24 C, 6, is greater than permitted
[19:00:12] SMILES Parse Error: extra open parentheses for input: 'CN(C1=C(=O)C=C2=C(N=C21)N3CC=CC(=N3)C4=CC=CC=C4'
[19:00:20] SMILES Parse Error: extra open parentheses for input: 'C[C@H](/C=C(\C)/C(=O)NO[C@H](C(=O)C(=O)N(C)C(=O)C(=O)N[C@@H](CCCC(=O)N([C@H](C(=O)N(C(=O)C(=O)N(C(=O)N1=O)CCC(=O)[C@@H](C2'
[19:00:20] SMILES Parse Error: extra open parentheses for input: 'C[C@H]1[C@@H]([C@H]([C@@H](O1)OC2=C(C(=O)C3=C2C=C(C=C3CCC(=O)C)C)CC4=CC=C(C=C4)O)O'
[19:00:20] Explicit valence for atom # 17 O, 3, is greater than permitted
[19:00:32] SMILES Parse Error: extra open parentheses for input: 'C[C@]12CCC(=O)C


[validation] Valid: 19/30 (63.3%)
[validation] Avg Tanimoto similarity: 0.711
[validation] Max similarity: 1.000

[generated molecules]
  1. ✗ C1=C(C=C(C=C1)C#N)C(C2=CC=C(C=C2)C#N3C=NC=N3)N               | sim: N/A
  2. ✓ C1=NC(=O)NC2=C1N=NC(=N2)C3=CC=CC=C3                          | sim: 0.208
  3. ✓ CC(=O)[O-].[Na+]                                             | sim: 1.000
  4. ✓ CN(C)CCN=C(N)N                                               | sim: 0.400
  5. ✓ C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2O)CCC4=C3C=CC(=C4)O     | sim: 1.000
  6. ✓ CN(C)CCN=C(N)C(=O)N                                          | sim: 0.308
  7. ✗ C1=CC=C(C(=C1O)O)C2=COC3=C2OC=C4C(=O)C=CC=C5                 | sim: N/A
  8. ✓ CC(C)(C)OO                                                   | sim: 1.000
  9. ✓ C[C@H](/C=C(\C)/C=C/C(=O)NO)C(=O)C1=CC=C(C=C1)N(C)C          | sim: 1.000
  10. ✗ CC(C)S(=O)(=O)NC1=C(C(=C(C=C1)F)C(=O)C2=CNC3=C2C=CC=C3C(=O)O | sim: N/A
  11. ✓ C(CCl)(Cl)(Cl)Cl                                  

...


[validation] Valid: 18/30 (60.0%)
[validation] Avg Tanimoto similarity: 0.473
[validation] Max similarity: 1.000

[generated molecules]
  1. ✓ C[C@H](/C=C(\C)/C=C/C(=O)NO)C(=O)C(=O)C1=CC=C=C(C=C1)N(C)C   | sim: 0.474
  2. ✓ C[C@H](/C=C/C(\C)/C)/C(=O)NOC(=O)C1=CC=C(C=C1)N(C)C          | sim: 0.389
  3. ✓ CN1CCC2=CC=CC=C21                                            | sim: 0.429
  4. ✗ CC(=O)O[C@H]1[C@H]([C@@H]2[C@]([C@H](CCC2(C)C)C)O)([C@@]3([C | sim: N/A
  5. ✗ C(C(C(S(=O)(=O)O)O)(O)O                                      | sim: N/A
  6. ✓ C1CCN(C1)C2=CC=CC=C2                                         | sim: 0.367
  7. ✗ C1=CC(=C(C=C1C2=C(C(=O)C3=C(C=C(C=C3O2)O)O)O)O               | sim: N/A
  8. ✗ C1=CC=C2C3=C4C(=C2=C1)C=CC5=C4C(=CC=C5)C=C3                  | sim: N/A
  9. ✓ CC(=O)NCCN                                                   | sim: 0.292
  10. ✓ C[C@]12C(=O)C[C@@H]1CC[C@@H]3[C@@H]2CC[C@]4([C@H]3CC[C@@H]4O | sim: 0.651
  11. ✓ C(C(C(F)(F)F)F)F                                    

[19:15:33] WARNING: not removing hydrogen atom without neighbors
[19:15:33] WARNING: not removing hydrogen atom without neighbors


[ref] Built 16029 reference fingerprints (bind=1)

[custom RNA] GUGAAAUGUUCAGGACCACUUG...


[19:16:02] SMILES Parse Error: unclosed ring for input: 'C1=CC=C2C(=C1)C(=CN2)S(=O)(=O)NCCC3=CC=CNC=C4'
[19:16:16] Explicit valence for atom # 2 C, 5, is greater than permitted
[19:16:20] SMILES Parse Error: extra open parentheses for input: 'C1=CC(=C(C=C1/C=C/C(=O)C(=O)O)/C=C/C(=C/C2=C(C=C(C=C2)O)O)O'
[19:16:20] SMILES Parse Error: extra open parentheses for input: 'C[C@]12CC[C@@H](C(=O)CC[C@@]1([C@@H](C(=O)C(=O)C(=O)O[C@H]2[C@@H]([C@H]([C@@H](C(=O)C)O[C@@H]3=CC=CC=C3C(=O)(C)O)C)O)OC(=O)O'
[19:16:24] SMILES Parse Error: extra open parentheses for input: 'C1=CC=C2C3=C4C(=C2=C1)C=CC(=C5=C4C(=CC=C5)C=C3'
[19:16:33] SMILES Parse Error: unclosed ring for input: 'CC(C)(C1=CC=C(C=C1)C)P(=S)(N2CCC2)CNCCNC(=O)S.CCCCCCN1'
[19:16:58] SMILES Parse Error: extra open parentheses for input: 'C/C(=O)OC1CC[N+]2(C1C(=O)C(C(=O)C(C2)C(=O)C(C)C(=O)OC(O)O)O'
[19:17:03] Explicit valence for atom # 3 N, 4, is greater than permitted
[19:17:07] SMILES Parse Error: unclosed ring for input: 'C1=CC=C(C=C1)NC2=CC=


[validation] Valid: 32/50 (64.0%)
[validation] Avg Tanimoto similarity: 0.598
[validation] Max similarity: 1.000

[results]
  ✗ INVALID | C1=CC=C2C(=C1)C(=CN2)S(=O)(=O)NCCC3=CC=CNC=C4                | sim: N/A
  ✓ VALID | C1=CC=C2C(=C1)C(=O)C3=CC(=C(C=C3O2)O)O                       | sim: 0.500
  ✓ VALID | CC1=C(C(=O)NC(=O)N1)F                                        | sim: 0.357
  ✓ VALID | C1=CC=C(C=C1)NC(=O)CCCCCC(=O)NO                              | sim: 1.000
  ✗ INVALID | C(C(C(F)(F)(F)F)(F)F)(F)F                                    | sim: N/A
  ✓ VALID | C([C@@H]1[C@@H]([C@H](C(O1)O)O)O)O                           | sim: 0.941
  ✗ INVALID | C1=CC(=C(C=C1/C=C/C(=O)C(=O)O)/C=C/C(=C/C2=C(C=C(C=C2)O)O)O  | sim: N/A
  ✗ INVALID | C[C@]12CC[C@@H](C(=O)CC[C@@]1([C@@H](C(=O)C(=O)C(=O)O[C@H]2[ | sim: N/A
  ✓ VALID | CN1CC[C@]23[C@@H]4[C@H]1CC5=C(C(=C5)O)O[C@H]2[C@H]3[C@H](C=C | sim: 0.422
  ✗ INVALID | C1=CC=C2C3=C4C(=C2=C1)C=CC(=C5=C4C(=CC=C5)C=C3               | sim: N/A
  ✗ INVALID | C